In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from encoder_decoder import*

# Playing around with encodings and embeddings to learn how to use them

In [2]:
# first import the little dictionary frol it_es_cognates.txt

italian_words = []
spanish_words = []

with open("it_es_cognates.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        it, es = line.split(";") # words separated by ;
        italian_words.append(it.lower())
        spanish_words.append(es.lower())

# check the number of words in the list of cognates
print(len(italian_words), "pairs")
print(italian_words[:5], spanish_words[:5])

# find all characters appearing in the list of cognates
all_chars = set()
for word in spanish_words + italian_words:
    all_chars.update(word)
all_chars = sorted(all_chars)

# add the special caracters: pad, start of string, end of string, unknown
specials = ['<pad>', '<sos>', '<eos>', '<unk>']
vocab = specials + all_chars

# build the dictionaries, just use the enumeration of vocab to assign an integer to every character
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# check the characters in the vocabulary, and its length
print("vocab: ", vocab)
print("vocab length:", len(vocab))

# test word size in the vocabulary (need to know the dimension of the words I am going to have in the model - by padding)
# I will add 2 or 3 just to be on the safe side for future additions to the dictionary, getting, say, to 20
print("Max length of spanish words: ", max(len(w) for w in spanish_words))
print("Max length of italian words: ", max(len(w) for w in italian_words))

def encode_source(word, max_len = 20):
    # input: no sos/eos needed
    ids = [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len] # use the char to int dictionary
    ids += [char_to_idx['<pad>']] * (max_len - len(ids)) # fill with '<pad>' until the prescribed length max_len
    return torch.tensor(ids)

def encode_target(word, max_len = 20):
    # output: needs sos/eos since decoder generates it step by step, they will replace two <pad> 
    ids = [char_to_idx['<sos>']] + [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len-2] + [char_to_idx['<eos>']]
    ids += [char_to_idx['<pad>']] * (max_len - len(ids))
    return torch.tensor(ids) 

def decode_source(ids):
    
    ids_numpy = ids.numpy()
    chars = []
    for i in ids_numpy:
        ch = idx_to_char[i] # use the integer to char dictionary
        if ch == '<pad>':
            break  # padding marks the end of real content
        chars.append(ch)
    return ''.join(chars)

# test
print(encode_source('castoro'))
print(decode_source(encode_source('castoro'))) 
print(decode_source(encode_source('è'))) # check unknown characters


# now test the embedding
vocab_size = len(char_to_idx)
d_model = 32 # needs to be large enough but does not have to be larger than number of characters - in fact for LLMs it is smaller than the number of tokens
embed = nn.Embedding(vocab_size, d_model, padding_idx=char_to_idx['<pad>'])

print(embed(encode_source('castoro')))

908 pairs
['acqua', 'aglio', 'aiutare', 'aiutata', 'aiutate'] ['agua', 'ajo', 'ayudar', 'ayudada', 'ayudadas']
vocab:  ['<pad>', '<sos>', '<eos>', '<unk>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', 'à', 'á', 'é', 'í', 'ñ', 'ó', 'ù', 'ú']
vocab length: 36
Max length of spanish words:  15
Max length of italian words:  13
tensor([ 6,  4, 21, 22, 17, 20, 17,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0])
castoro
<unk>
tensor([[-2.9588e-02,  7.0263e-01,  2.2844e+00,  1.0320e+00,  9.0779e-01,
         -7.6287e-01, -8.0142e-01,  4.9292e-01,  9.1089e-01,  1.4039e+00,
          5.4536e-02,  5.8771e-01,  3.6513e-01, -2.8289e-01,  1.8509e-01,
         -5.1451e-01,  7.6824e-01, -1.7176e+00,  9.5445e-01,  3.5613e-02,
          1.1873e-01, -1.1700e+00, -1.4976e+00, -4.5592e-01,  2.0375e-01,
         -6.3999e-01, -2.3467e-01,  2.2417e-01,  1.8527e+00, -5.7445e-01,
         -1.5282e+00, -1.7670e-01],
        [-

In [13]:
# test if Encoder runs correctly

enco = Encoder(2, d_model = d_model, d_hidden = 4*d_model, dk = 8, dv = 8, h = 4)
pos_enco = PositionalEncodingModule(d_model=d_model)

encoded_word = pos_enco(enco(embed(encode_source('castoro')).unsqueeze(0)))
# print(encoded_word)

# ok

# test if Transformer runs correctly 

trafo = Transformer(d_model = d_model, vocab_size = vocab_size, char_to_idx = char_to_idx, dk = 8, dv = 8)

print(trafo(encode_source("castoro").unsqueeze(0), encode_source("diga").unsqueeze(0)))
print(trafo.generate_sequence(encode_source("castoro").unsqueeze(0)))

tensor([[[-2.6374e-01, -5.8479e-02,  3.9823e-01,  5.0421e-01, -1.8865e-01,
          -6.3954e-02,  3.5832e-01,  5.3442e-02,  3.7729e-01,  1.3084e-01,
           3.0413e-01, -1.0142e+00, -1.9297e-01, -4.3899e-01,  1.0076e+00,
          -1.0426e+00,  1.7512e-01,  6.9688e-01, -2.2547e-01,  8.9988e-02,
           6.2843e-01,  2.3131e-01,  3.9812e-01, -5.4482e-01, -7.5827e-01,
          -4.6570e-01,  6.8174e-01, -5.5388e-01,  6.5539e-02,  3.8061e-01,
           4.3103e-01,  3.1205e-02,  3.1950e-01, -4.5393e-01,  5.4794e-02,
           9.8810e-02],
         [-1.3036e+00, -6.6842e-01, -1.6624e-02, -3.9383e-01, -3.4021e-01,
          -1.2399e-01, -1.1149e-01, -4.3173e-01, -5.8649e-01, -9.0984e-01,
           2.0170e-01,  7.0976e-02,  7.3818e-01, -1.3056e-01,  1.2928e-01,
          -1.8115e-01,  2.6068e-01,  2.8508e-01, -3.7332e-01,  7.2892e-01,
          -3.6803e-01,  4.9794e-02, -1.1544e-01, -7.4714e-02, -5.9906e-01,
           9.4111e-02,  8.6642e-01,  1.9608e-02,  7.5627e-01,  6.8021e-01,
 